# Simple CUDA Metrics Demo (Colab)

Goal: compare a tiny CPU program vs a tiny CUDA C++ program on the same synthetic project-style data.

We compute only 4 metrics so every line is easy to explain:
- total_rows
- success_count
- score_sum
- max_score

Class script (60s):
1. Cell 2 checks we really have a GPU runtime.
2. Cell 3 loads the synthetic CSV.
3. Cells 4 and 5 write CPU and GPU source code.
4. Cell 6 compiles and runs both binaries.
5. Cell 7 confirms outputs match (correctness).
6. Cell 8 measures timing and reports speedup (performance).

In [ ]:
%%bash
set -e
echo "Checking GPU runtime..."
nvidia-smi
echo
echo "Checking nvcc..."
nvcc --version

In [ ]:
from pathlib import Path
import pandas as pd

csv_path = Path('synthetic_metrics.csv')
if not csv_path.exists():
    raise FileNotFoundError('synthetic_metrics.csv not found. Upload this file into the same Colab working directory.')

df = pd.read_csv(csv_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
display(df.head(10))

In [ ]:
%%writefile cpu_metrics.cpp
#include <algorithm>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <sstream>
#include <string>

int main(int argc, char** argv) {
    if (argc < 2) {
        std::cerr << "Usage: ./cpu_metrics <csv_file>\n";
        return 1;
    }

    std::ifstream file(argv[1]);
    if (!file.is_open()) {
        std::cerr << "Could not open file: " << argv[1] << "\n";
        return 1;
    }

    std::string line;
    std::getline(file, line); // skip header

    long long total_rows = 0;
    long long success_count = 0;
    double score_sum = 0.0;
    double max_score = 0.0;

    while (std::getline(file, line)) {
        if (line.empty()) continue;
        std::stringstream ss(line);
        std::string token;

        std::getline(ss, token, ','); // harm_category_index
        std::getline(ss, token, ','); // final_score
        double score = std::stod(token);
        std::getline(ss, token, ','); // iterations
        std::getline(ss, token, ','); // success_flag
        int success = std::stoi(token);

        total_rows += 1;
        success_count += success;
        score_sum += score;
        max_score = std::max(max_score, score);
    }

    std::cout << std::fixed << std::setprecision(6);
    std::cout << "total_rows=" << total_rows << "\n";
    std::cout << "success_count=" << success_count << "\n";
    std::cout << "score_sum=" << score_sum << "\n";
    std::cout << "max_score=" << max_score << "\n";
    return 0;
}

In [ ]:
%%writefile gpu_metrics.cu
#include <cuda_runtime.h>

#include <algorithm>
#include <cstdlib>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <sstream>
#include <string>
#include <vector>

#define CUDA_CHECK(call) do { \
  cudaError_t err = (call); \
  if (err != cudaSuccess) { \
    std::cerr << "CUDA error: " << cudaGetErrorString(err) << "\n"; \
    std::exit(1); \
  } \
} while (0)

__device__ float atomicMaxFloat(float* address, float val) {
  int* address_as_i = reinterpret_cast<int*>(address);
  int old = *address_as_i, assumed;
  while (__int_as_float(old) < val) {
    assumed = old;
    old = atomicCAS(address_as_i, assumed, __float_as_int(val));
    if (assumed == old) break;
  }
  return __int_as_float(old);
}

__global__ void metrics_kernel(const float* scores, const int* successes, int n,
                               long long* total_rows, long long* success_count,
                               float* score_sum, float* max_score) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i >= n) return;

  atomicAdd(total_rows, 1LL);
  atomicAdd(success_count, static_cast<long long>(successes[i]));
  atomicAdd(score_sum, scores[i]);
  atomicMaxFloat(max_score, scores[i]);
}

int main(int argc, char** argv) {
  if (argc < 2) {
    std::cerr << "Usage: ./gpu_metrics <csv_file>\n";
    return 1;
  }

  std::ifstream file(argv[1]);
  if (!file.is_open()) {
    std::cerr << "Could not open file: " << argv[1] << "\n";
    return 1;
  }

  std::string line;
  std::getline(file, line); // skip header

  std::vector<float> h_scores;
  std::vector<int> h_successes;

  while (std::getline(file, line)) {
    if (line.empty()) continue;
    std::stringstream ss(line);
    std::string token;

    std::getline(ss, token, ','); // harm_category_index
    std::getline(ss, token, ','); // final_score
    float score = std::stof(token);
    std::getline(ss, token, ','); // iterations
    std::getline(ss, token, ','); // success_flag
    int success = std::stoi(token);

    h_scores.push_back(score);
    h_successes.push_back(success);
  }

  int n = static_cast<int>(h_scores.size());
  if (n == 0) {
    std::cerr << "No data rows found\n";
    return 1;
  }

  float* d_scores = nullptr;
  int* d_successes = nullptr;
  long long* d_total_rows = nullptr;
  long long* d_success_count = nullptr;
  float* d_score_sum = nullptr;
  float* d_max_score = nullptr;

  CUDA_CHECK(cudaMalloc(&d_scores, n * sizeof(float)));
  CUDA_CHECK(cudaMalloc(&d_successes, n * sizeof(int)));
  CUDA_CHECK(cudaMalloc(&d_total_rows, sizeof(long long)));
  CUDA_CHECK(cudaMalloc(&d_success_count, sizeof(long long)));
  CUDA_CHECK(cudaMalloc(&d_score_sum, sizeof(float)));
  CUDA_CHECK(cudaMalloc(&d_max_score, sizeof(float)));

  CUDA_CHECK(cudaMemcpy(d_scores, h_scores.data(), n * sizeof(float), cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_successes, h_successes.data(), n * sizeof(int), cudaMemcpyHostToDevice));

  CUDA_CHECK(cudaMemset(d_total_rows, 0, sizeof(long long)));
  CUDA_CHECK(cudaMemset(d_success_count, 0, sizeof(long long)));
  CUDA_CHECK(cudaMemset(d_score_sum, 0, sizeof(float)));
  CUDA_CHECK(cudaMemset(d_max_score, 0, sizeof(float)));

  int block_size = 256;
  int grid_size = (n + block_size - 1) / block_size;
  metrics_kernel<<<grid_size, block_size>>>(d_scores, d_successes, n, d_total_rows, d_success_count, d_score_sum, d_max_score);
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaDeviceSynchronize());

  long long total_rows = 0;
  long long success_count = 0;
  float score_sum = 0.0f;
  float max_score = 0.0f;

  CUDA_CHECK(cudaMemcpy(&total_rows, d_total_rows, sizeof(long long), cudaMemcpyDeviceToHost));
  CUDA_CHECK(cudaMemcpy(&success_count, d_success_count, sizeof(long long), cudaMemcpyDeviceToHost));
  CUDA_CHECK(cudaMemcpy(&score_sum, d_score_sum, sizeof(float), cudaMemcpyDeviceToHost));
  CUDA_CHECK(cudaMemcpy(&max_score, d_max_score, sizeof(float), cudaMemcpyDeviceToHost));

  CUDA_CHECK(cudaFree(d_scores));
  CUDA_CHECK(cudaFree(d_successes));
  CUDA_CHECK(cudaFree(d_total_rows));
  CUDA_CHECK(cudaFree(d_success_count));
  CUDA_CHECK(cudaFree(d_score_sum));
  CUDA_CHECK(cudaFree(d_max_score));

  std::cout << std::fixed << std::setprecision(6);
  std::cout << "total_rows=" << total_rows << "\n";
  std::cout << "success_count=" << success_count << "\n";
  std::cout << "score_sum=" << score_sum << "\n";
  std::cout << "max_score=" << max_score << "\n";
  return 0;
}

In [ ]:
%%bash
set -e
echo "Compiling CPU program..."
g++ -O2 -std=c++17 cpu_metrics.cpp -o cpu_metrics
echo "Compiling CUDA program..."
nvcc -O2 -std=c++17 gpu_metrics.cu -o gpu_metrics
echo
echo "CPU output:"
./cpu_metrics synthetic_metrics.csv | tee cpu_output.txt
echo
echo "GPU output:"
./gpu_metrics synthetic_metrics.csv | tee gpu_output.txt

In [ ]:
from pathlib import Path

def parse_metrics(text):
    out = {}
    for line in text.strip().splitlines():
        k, v = line.split('=')
        out[k.strip()] = float(v.strip())
    return out

cpu_metrics = parse_metrics(Path('cpu_output.txt').read_text(encoding='utf-8'))
gpu_metrics = parse_metrics(Path('gpu_output.txt').read_text(encoding='utf-8'))

eps = 1e-4
checks = {
    'total_rows': abs(cpu_metrics['total_rows'] - gpu_metrics['total_rows']) == 0,
    'success_count': abs(cpu_metrics['success_count'] - gpu_metrics['success_count']) == 0,
    'score_sum': abs(cpu_metrics['score_sum'] - gpu_metrics['score_sum']) <= eps,
    'max_score': abs(cpu_metrics['max_score'] - gpu_metrics['max_score']) <= eps,
}

for name, ok in checks.items():
    print(f'{name}:', 'PASS' if ok else 'FAIL')

all_ok = all(checks.values())
print('\nParity result:', 'PASS' if all_ok else 'FAIL')
if not all_ok:
    raise AssertionError('CPU and GPU results do not match.')

In [ ]:
%%bash
set -e
head -n 1 synthetic_metrics.csv > synthetic_metrics_big.csv
for i in $(seq 1 12000); do
  tail -n +2 synthetic_metrics.csv >> synthetic_metrics_big.csv
done
rows=$(($(wc -l < synthetic_metrics_big.csv)-1))
echo "Big dataset rows: $rows"
echo
cpu_times=()
gpu_times=()
for i in $(seq 1 6); do
  t=$( (/usr/bin/time -f "%e" ./cpu_metrics synthetic_metrics_big.csv > /dev/null) 2>&1 )
  cpu_times+=("$t")
done
for i in $(seq 1 6); do
  t=$( (/usr/bin/time -f "%e" ./gpu_metrics synthetic_metrics_big.csv > /dev/null) 2>&1 )
  gpu_times+=("$t")
done
cpu_avg=$(printf "%s\n" "${cpu_times[@]}" | awk '{sum += $1} END {if (NR > 0) printf "%.6f", sum / NR; else print "0"}')
gpu_avg=$(printf "%s\n" "${gpu_times[@]}" | awk '{sum += $1} END {if (NR > 0) printf "%.6f", sum / NR; else print "0"}')
speedup=$(awk -v c="$cpu_avg" -v g="$gpu_avg" 'BEGIN {if (g > 0) printf "%.2f", c / g; else print "inf"}')
echo "CPU times: ${cpu_times[*]}"
echo "GPU times: ${gpu_times[*]}"
echo "Average CPU time: ${cpu_avg} s"
echo "Average GPU time: ${gpu_avg} s"
echo "Speedup (CPU/GPU): ${speedup}x"
echo
echo "GPU utilization snapshot:"
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv